# Sensor Consistency Models — Screening & Threshold Calibration

Steps 1-2 of the fault-detection stage: for each sensor, train a simple model that predicts it from every *other* sensor plus the two operating settings, using only clean training engines. Screen out sensors the model can't predict well (a residual on an unpredictable sensor is noise, not a fault signal), then calibrate a normalized threshold and a persistence rule on the validation engines.

RUL is never used here, by design -- it isn't available at deployment time, and including it would let the model quietly explain away real drift as normal aging instead of flagging it.

Capped vs. uncapped RUL makes no difference in this notebook, since RUL is never touched as input or output -- `processed_train.csv` is used purely for `Engine_ID`, `Cycle`, and the sensor/setting columns.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import joblib

In [2]:
df = pd.read_csv("Dataset/processed_train.csv")

np.random.seed(42)

engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = df[df["Engine_ID"].isin(train_ids)].copy()
val_df = df[df["Engine_ID"].isin(val_ids)].copy()
test_df = df[df["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())

Training engines: 70
Validation engines: 15
Testing engines: 15


In [3]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

# Only actual sensors get monitored for faults -- Setting_1/Setting_2 are
# operating conditions, always used as *inputs*, never as a target to
# validate.
SENSOR_COLS = [f for f in FEATURES if f.startswith("Sensor")]
print("Candidate sensors to screen:", len(SENSOR_COLS))

Candidate sensors to screen: 15


## Step 1 — Train one consistency model per sensor, screen by R²

In [4]:
R2_SCREEN_THRESHOLD = 0.5  # keep sensors the model can actually predict

models = {}
r2_scores = {}

for target in SENSOR_COLS:
    predictors = [f for f in FEATURES if f != target]

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model.fit(train_df[predictors], train_df[target])

    val_pred = model.predict(val_df[predictors])
    r2 = r2_score(val_df[target], val_pred)

    models[target] = model
    r2_scores[target] = r2

screening = pd.DataFrame({
    "Sensor": list(r2_scores.keys()),
    "R2_on_val": list(r2_scores.values())
}).sort_values("R2_on_val", ascending=False).reset_index(drop=True)

screening["Monitor"] = screening["R2_on_val"] >= R2_SCREEN_THRESHOLD
print(screening)

       Sensor  R2_on_val  Monitor
0    Sensor_9   0.893953     True
1   Sensor_14   0.893271     True
2   Sensor_11   0.840328     True
3   Sensor_12   0.816161     True
4    Sensor_8   0.786378     True
5   Sensor_13   0.783598     True
6    Sensor_4   0.779212     True
7    Sensor_7   0.778398     True
8   Sensor_15   0.709177     True
9   Sensor_20   0.684246     True
10  Sensor_21   0.672966     True
11  Sensor_17   0.621976     True
12   Sensor_2   0.620318     True
13   Sensor_3   0.559559     True
14   Sensor_6   0.049182    False


In [5]:
MONITORED_SENSORS = screening.loc[screening["Monitor"], "Sensor"].tolist()
DROPPED_SENSORS = screening.loc[~screening["Monitor"], "Sensor"].tolist()

print(f"Monitoring {len(MONITORED_SENSORS)} of {len(SENSOR_COLS)} sensors:")
print(MONITORED_SENSORS)
print(f"\nDropped ({len(DROPPED_SENSORS)}) -- model can't predict these well enough to trust a residual on them:")
print(DROPPED_SENSORS)

Monitoring 14 of 15 sensors:
['Sensor_9', 'Sensor_14', 'Sensor_11', 'Sensor_12', 'Sensor_8', 'Sensor_13', 'Sensor_4', 'Sensor_7', 'Sensor_15', 'Sensor_20', 'Sensor_21', 'Sensor_17', 'Sensor_2', 'Sensor_3']

Dropped (1) -- model can't predict these well enough to trust a residual on them:
['Sensor_6']


## Step 2 — Calibrate a normalized threshold per monitored sensor on `val_df`

In [6]:
PERCENTILE = 99

sigmas = {}
thresholds = {}

for target in MONITORED_SENSORS:
    predictors = [f for f in FEATURES if f != target]
    val_pred = models[target].predict(val_df[predictors])

    residual = np.abs(val_df[target].values - val_pred)
    sigma = residual.std()

    normalized_residual = residual / sigma
    threshold = np.percentile(normalized_residual, PERCENTILE)

    sigmas[target] = sigma
    thresholds[target] = threshold

calibration = pd.DataFrame({
    "Sensor": MONITORED_SENSORS,
    "Residual_std_clean": [sigmas[s] for s in MONITORED_SENSORS],
    "Threshold_99th_pct": [thresholds[s] for s in MONITORED_SENSORS]
})
print(calibration)

       Sensor  Residual_std_clean  Threshold_99th_pct
0    Sensor_9            0.013993            4.266150
1   Sensor_14            0.013861            4.264896
2   Sensor_11            0.040243            4.330870
3   Sensor_12            0.042099            4.284852
4    Sensor_8            0.030277            4.296359
5   Sensor_13            0.029455            4.256753
6    Sensor_4            0.043828            4.271725
7    Sensor_7            0.041678            4.264315
8   Sensor_15            0.048408            4.300721
9   Sensor_20            0.047626            4.139222
10  Sensor_21            0.051710            4.093260
11  Sensor_17            0.048266            4.376500
12   Sensor_2            0.056534            4.233200
13   Sensor_3            0.053809            4.302786


## Persistence rule — a single bad cycle shouldn't trigger a fault

In [7]:
def check_persistence(flags, window=7, min_count=5):
    """flags: boolean array/series, True where the residual exceeded
    threshold that cycle. Returns a boolean array, True at position i if
    the fault condition held for at least min_count of the last `window`
    cycles (including position i)."""
    flags = np.asarray(flags).astype(int)
    rolling_sum = pd.Series(flags).rolling(window, min_periods=1).sum().values
    return rolling_sum >= min_count

PERSISTENCE_WINDOW = 7
PERSISTENCE_MIN_COUNT = 5

## Sanity check — false-alarm rate on clean validation data

Before moving to fault injection: verify the calibrated threshold + persistence rule doesn't already flag clean engines it's supposed to consider normal.

In [8]:
false_alarms = {}

for target in MONITORED_SENSORS:
    predictors = [f for f in FEATURES if f != target]

    alarm_any_engine = []
    for engine_id, engine_data in val_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        pred = models[target].predict(engine_data[predictors])
        residual = np.abs(engine_data[target].values - pred) / sigmas[target]
        flags = residual > thresholds[target]
        persistent = check_persistence(flags, PERSISTENCE_WINDOW, PERSISTENCE_MIN_COUNT)
        alarm_any_engine.append(persistent.any())

    false_alarms[target] = np.mean(alarm_any_engine)

fa_df = pd.DataFrame({
    "Sensor": MONITORED_SENSORS,
    "False_alarm_rate_on_clean_val_engines": [false_alarms[s] for s in MONITORED_SENSORS]
})
print(fa_df)
print("\nIf any of these are high, tighten the threshold or persistence rule for that sensor before trusting its detections.")

       Sensor  False_alarm_rate_on_clean_val_engines
0    Sensor_9                                    0.0
1   Sensor_14                                    0.0
2   Sensor_11                                    0.0
3   Sensor_12                                    0.0
4    Sensor_8                                    0.0
5   Sensor_13                                    0.0
6    Sensor_4                                    0.0
7    Sensor_7                                    0.0
8   Sensor_15                                    0.0
9   Sensor_20                                    0.0
10  Sensor_21                                    0.0
11  Sensor_17                                    0.0
12   Sensor_2                                    0.0
13   Sensor_3                                    0.0

If any of these are high, tighten the threshold or persistence rule for that sensor before trusting its detections.


## Save everything for the fault-injection notebook

In [9]:
joblib.dump({
    "models": models,
    "sigmas": sigmas,
    "thresholds": thresholds,
    "monitored_sensors": MONITORED_SENSORS,
    "features": FEATURES,
    "persistence_window": PERSISTENCE_WINDOW,
    "persistence_min_count": PERSISTENCE_MIN_COUNT
}, "consistency_models.pkl")

print("Saved consistency_models.pkl")

Saved consistency_models.pkl
